# Inspiration Behind the Project
As a developer, I deal with SSH keys on a daily basis as it is fundamental to access servers and manage files/processes.

PuTTYgen is an SSH key generator and manager that allows people to generate public-private key pairs, convert private keys into public keys and add comments to keys. This is the exact logic I wanted to replicate. MacOS doesn't natively allow PuTTYgen to be installed, so I wanted to create something similar to it, hence MacPuTTY.

# Initial Logic
The first thing to do is create the main logic. Using Python for the core logic tied with a Flask backend for proper routing, I began the first journey of setting up the base logic: creating a public-private key pair. There are various types of cryptographical keys that could be used, but I decided to use the main 3 that I was familiar with: RSA keys, ECDSA keys and ED25519 keys.

Using the `cryptography` library, I was able to set up the first few functions to generate these key pairs.

In [ ]:
from cryptography.hazmat.primitives import serialization
from cryptography.hazmat.primitives.serialization import Encoding, PrivateFormat, PublicFormat, NoEncryption

# rsa imports
from cryptography.hazmat.primitives.asymmetric import rsa

# ecdsa imports
from cryptography.hazmat.primitives.asymmetric import ec
from cryptography.hazmat.primitives import hashes

# ed25519 import
from cryptography.hazmat.primitives.asymmetric import ed25519

These imports would allow be to be able to set up the creation of keys. After importing them, I began starting to create the actual functions that would generate the keys for me.

In [ ]:
def generate_rsa_key_pair():
    # Generate a private RSA key
    private_key = rsa.generate_private_key(
        public_exponent=65537,
        key_size=rsa_key_bits,
    )

    # Create a public key via the private RSA key
    public_key = private_key.public_key()


    # File and directory information where the files will be placed
    key_directory = Path(ssh_directory).expanduser() # using HOCON formatting to set this
    key_directory.mkdir(parents=True, exist_ok=True)

    private_rsa_key_filename = "id_rsa"
    public_rsa_key_filename = "id_rsa.pub"

    private_key_filepath = key_directory / private_rsa_key_filename
    public_key_filepath = key_directory / public_rsa_key_filename

    # Private and public PEM generation
    private_pem = private_key.private_bytes(
        encoding=serialization.Encoding.PEM,
        format=serialization.PrivateFormat.OpenSSH,
        encryption_algorithm=serialization.NoEncryption(),
    )

    public_pem = public_key.public_bytes(
        encoding=serialization.Encoding.OpenSSH,
        format=serialization.PublicFormat.OpenSSH,
    )

    # Create the actual key files
    private_key_filepath.write_bytes(private_pem)
    public_key_filepath.write_bytes(public_pem)

    # Set up correct permissions
    private_key_filepath.chmod(0o600)

This function would allow be to be able to generate a private and public RSA key pair. Standardizing the application with a HOCON config allows the user and myself to be able to set up the directory where the SSH keys live (typically `~/.ssh` on MacOS). Similar functions for ECDSA and ED25519 key-pair generation was set up.

The rest was just adding additional features and using the same logic to read files, generate public keys with a private key, and combining everything all together with Flask routes.

# Setting Up the Initial MVP
I decided to containerize the project using Docker Compose. The process was really simple:
1. Create `.dockerignore`, a `Dockerfile` and a `docker-compose.yml`
2. Use port `5050` rather than `5000` due to the port being in use (may not be the case for everyone)
3. Run `docker compose up --build -d` to get the backend running

For the frontend, I used Electron so it would appear as an application, so it just involved changing into the `ui/` directory:
```bash
    cd ui && npm install
	cd ui && npm start
```

And boom, the initial MVP was done! I added more features to improve quality of life and to improve the overall app.